# Natural Language Processing Project - Menganalisis Dampak Terhadap Nilai Tukar Dolar
Arranged by:
*   Anders Emmanuel Tan (24/541351/PA/22964)
*   Azhar Maulana (24/533487/PA/22582)
*   Evan Razzan Adytaputra (24/545257/PA/23166)
*   Kukuh Agus Hermawan (24/533395/PA/22573)

Deskripsi Modul: Notebook ini berfungsi sebagai pengumpul data berita geopolitik secara otomatis.

## 0. Instalasi Library pendukung
Description: Cell dibawah ini berfungsi untuk menginstal beberapa library Python yang dibutuhkan untuk *web scraping*, mengambil berita, parsing HTML, dan memproses data.

In [1]:
%pip install -q trafilatura googlenewsdecoder beautifulsoup4 pandas requests

Note: you may need to restart the kernel to use updated packages.


## 1. Pengaturan Parameter Scraping dan Penyaring Kata Kunci

Bagian ini mengatur bagaimana pengambilan berita akan dijalankan:
* **Rentang Waktu**: Mengambil data dari September 2021 hingga September 2026.
* **Interval**: Mengambil data setiap 14 hari sekali dengan durasi pengamatan selama 4 hari.
* **Penyaringan Kata Kunci (Filtering)**:
  * **Diabaikan (`EXCLUDE_KEYWORDS`)**: Menghapus berita seputar hiburan, kripto, gaya hidup, atau tips keuangan pribadi agar data tetap fokus pada geopolitik.
  * **Diutamakan (`CORE_GEOPOLITICAL_KEYWORDS`)**: Hanya menyimpan berita yang mengandung isu geopolitik, kebijakan bank sentral, suku bunga, atau perang.

In [ ]:
import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
import trafilatura
from googlenewsdecoder import gnewsdecoder
from bs4 import BeautifulSoup

START_DATE = datetime(2021, 9, 1)
END_DATE = datetime(2026, 9, 1)

SAMPLE_INTERVAL_DAYS = 14     # Bi-weekly intervals for uniform temporal distribution
MAX_PER_INTERVAL = 20

OUTPUT_DIR = os.path.join("..", "data/raw")
OUTPUT_CSV = "geopolitical_news.csv"
SAVE_PATH = os.path.join(OUTPUT_DIR, OUTPUT_CSV)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Strategic Filtering: Drop consumer finance, crypto, entertainment, and lifestyle
EXCLUDE_KEYWORDS = [
    "nft", "meme", "doge", "crypto", "bitcoin", "ethereum", "token", 
    "dating", "wedding", "mansion", "celebrity", "movie", "box office",
    "tips for", "how to beat", "how to save", "credit score", "mortgage rate",
    "retirement", "best stocks", "undervalued"
]

# Strategic Filtering: Require sovereign macroeconomic and geopolitical signals[cite: 1]
CORE_GEOPOLITICAL_KEYWORDS = [
    "sanction", "sanctions", "tariff", "tariffs", "trade war", "embargo",
    "federal reserve", "central bank", "bank indonesia", "monetary policy",
    "rate hike", "rate cut", "interest rate", "foreign exchange", "forex", 
    "currency", "dollar index", "usd/idr", "treasury yield", "geopolitics", 
    "geopolitical", "foreign reserves", "sovereign debt", "war", "military"
]

print(f"Configuration set. Target file: {SAVE_PATH}")

Configuration set. Target file: ..\data/raw\geopolitical_news.csv


## 3. Fungsi Pembantu Ekstraksi Isi Berita dan Penyaring Otomatis

Di sini dibuat dua fungsi utama:
1. `parse_body()`: Mengambil teks utama/artikel dari halaman web HTML secara bersih dan membuang elemen yang tidak penting.
2. `passes_strategic_filter()`: Memeriksa apakah suatu berita layak disimpan. Berita akan dibuang jika mengandung kata-kata yang diabaikan dan hanya diterima jika memiliki minimal 1 indikator geopolitik/ekonomi utama pada judul atau paragraf awal.

In [3]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
})

def parse_body(html_text: str) -> str:
    """Extracts readable article content with fallback to standard paragraph tags."""
    text = trafilatura.extract(html_text, include_comments=False, include_tables=False)
    if text and len(text.strip()) > 150:
        return text.strip()
    soup = BeautifulSoup(html_text, "html.parser")
    paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 40]
    extracted = " ".join(paragraphs)
    return extracted.strip() if len(extracted) > 150 else ""

def passes_strategic_filter(title: str, content: str) -> bool:
    """Applies negative noise suppression and positive macroeconomic relevance gating[cite: 1]."""
    title_lower = title.lower()
    content_lower = content.lower()
    full_text = title_lower + " " + content_lower

    # 1. Negative Noise Filtering[cite: 1]
    for bad_word in EXCLUDE_KEYWORDS:
        if bad_word in title_lower:
            return False

    # 2. Positive Filtering on title and lead text[cite: 1]
    lead_text = full_text[:1200]
    matched_signals = sum(1 for signal in CORE_GEOPOLITICAL_KEYWORDS if signal in lead_text)
    return matched_signals >= 1

## 4. Sistem Checkpoint

Kode ini memeriksa apakah sudah ada file CSV dari proses *scraping* sebelumnya:
* Jika file sudah ada, sistem akan membaca berita yang sudah pernah diunduh agar **tidak terjadi duplikasi**.
* Sistem akan melewati tanggal-tanggal yang sudah selesai dan melanjutkan *scraping* untuk tanggal yang belum terproses.
Hal tersebut dilakukan agar pengembang tidak harus mengulang proses scraping jika koneksi internet hilang.

In [4]:
# Cell 4: Check for previous CSV and identify completed checkpoints
checkpoints = []
curr = START_DATE
while curr <= END_DATE:
    checkpoints.append(curr)
    curr += timedelta(days=SAMPLE_INTERVAL_DAYS)

# Load past CSV data if it exists
if os.path.exists(SAVE_PATH):
    df_existing = pd.read_csv(SAVE_PATH)
    processed_titles = set(df_existing["title"].dropna().tolist())
    records = df_existing.to_dict("records")
    
    # Extract dates that have already been collected
    if not df_existing.empty and "published_at" in df_existing.columns:
        df_existing["published_at"] = pd.to_datetime(df_existing["published_at"])
        # A checkpoint is considered finished if we already have records from its time window
        completed_dates = {d.strftime("%Y-%m-%d") for d in df_existing["published_at"].dt.date}
    else:
        completed_dates = set()
        
    print(f"Resuming previous progress: Loaded {len(records)} articles from {SAVE_PATH}.")
else:
    processed_titles = set()
    records = []
    completed_dates = set()
    print("No previous progress found. Starting from scratch.")

print(f"Total target checkpoints: {len(checkpoints)}")

No previous progress found. Starting from scratch.
Total target checkpoints: 131


## 5. Pelaksanaan Scraping Berita

Proses pengambilan data utama dilakukan pada bagian ini:
* **Multi-threading**: Menggunakan 16 *worker* sekaligus untuk mempercepat proses *download* artikel.
* **Pencarian Berita**: Mengambil berita dari Google News RSS dengan sumber yang dicantumkan ditugas (seperti CNBC).
* **Penyimpanan Otomatis**: Setiap kali satu titik waktu selesai diproses, hasilnya langsung disimpan ke dalam file CSV agar data tidak hilang jika koneksi terputus di tengah jalan.

In [ ]:
# Cell 5: Multi-threaded scraping with instant saving & checkpoint skipping
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_single_item(item, dt_start):
    """Processes, decodes, and parses a single news item."""
    title_full = item.find("title").text if item.find("title") is not None else ""
    rss_link = item.find("link").text if item.find("link") is not None else ""
    pub_date = item.find("pubDate").text if item.find("pubDate") is not None else ""
    source_elem = item.find("source")
    source_name = source_elem.text if source_elem is not None else "Unknown"
    title_clean = title_full.rsplit(" - ", 1)[0] if " - " in title_full else title_full

    if not title_clean or title_clean in processed_titles:
        return None
    if any(bad in title_clean.lower() for bad in EXCLUDE_KEYWORDS):
        return None

    try:
        dt_parsed = pd.to_datetime(pub_date)
    except Exception:
        dt_parsed = dt_start

    # Resolve redirected Google News URL
    actual_url = rss_link
    try:
        decoded = gnewsdecoder(rss_link, interval=0.1)
        if isinstance(decoded, dict) and decoded.get("status"):
            actual_url = decoded.get("decoded_url", rss_link)
    except Exception:
        pass

    # Extract clean article body
    content_text = ""
    try:
        art_resp = session.get(actual_url, timeout=8)
        if art_resp.status_code == 200:
            content_text = parse_body(art_resp.text)
    except Exception:
        pass

    # Validate article content
    if len(content_text) > 200 and passes_strategic_filter(title_clean, content_text):
        return {
            "published_at": dt_parsed,
            "title": title_clean,
            "content": content_text,
            "source_domain": source_name,
            "language": "en",
            "url": actual_url
        }
    return None


for idx, dt_start in enumerate(checkpoints, 1):
    dt_end = dt_start + timedelta(days=SAMPLE_INTERVAL_DAYS)
    after_str = dt_start.strftime("%Y-%m-%d")
    before_str = dt_end.strftime("%Y-%m-%d")

    # Skip this checkpoint if we already have articles from this date range
    if after_str in completed_dates:
        print(f"[{idx}/{len(checkpoints)}] {after_str}: Already scraped. Skipping...")
        continue

    query = (
        f'(dollar OR USD OR "Federal Reserve" OR tariff OR sanctions OR inflation OR war) '
        f'(site:cnbc.com) '
        f'after:{after_str} before:{before_str}'
    )
    rss_url = f"https://news.google.com/rss/search?q={requests.utils.quote(query)}&hl=en-US&gl=US&ceid=US:en"

    try:
        resp = session.get(rss_url, timeout=12)
        if resp.status_code == 200:
            root = ET.fromstring(resp.content)
            items = root.findall(".//item")[:MAX_PER_INTERVAL]

            new_articles = []
            # Concurrently fetch articles with 16 workers
            with ThreadPoolExecutor(max_workers=16) as executor:
                futures = [executor.submit(process_single_item, it, dt_start) for it in items]
                for future in as_completed(futures):
                    result = future.result()
                    if result and result["title"] not in processed_titles:
                        new_articles.append(result)
                        processed_titles.add(result["title"])

            if new_articles:
                records.extend(new_articles)
                completed_dates.add(after_str)

            # SAVE IMMEDIATELY to disk on every checkpoint iteration
            pd.DataFrame(records).to_csv(SAVE_PATH, index=False)

            print(f"[{idx}/{len(checkpoints)}] {after_str} to {before_str}: Saved {len(new_articles)} new articles (Total in CSV: {len(records)})")
        else:
            print(f"[{idx}/{len(checkpoints)}] RSS Error: HTTP {resp.status_code}")

    except Exception as e:
        print(f"[{idx}/{len(checkpoints)}] Checkpoint Error: {e}")

    time.sleep(0.4)

print("\nAll checkpoints processed!")

[1/131] 2021-09-01 to 2021-09-15: Saved 13 new articles (Total in CSV: 13)
[2/131] 2021-09-15 to 2021-09-29: Saved 12 new articles (Total in CSV: 25)
[3/131] 2021-09-29 to 2021-10-13: Saved 9 new articles (Total in CSV: 34)
[4/131] 2021-10-13 to 2021-10-27: Saved 12 new articles (Total in CSV: 46)
[5/131] 2021-10-27 to 2021-11-10: Saved 9 new articles (Total in CSV: 55)
[6/131] 2021-11-10 to 2021-11-24: Saved 10 new articles (Total in CSV: 65)
[7/131] 2021-11-24 to 2021-12-08: Saved 13 new articles (Total in CSV: 78)
[8/131] 2021-12-08 to 2021-12-22: Saved 8 new articles (Total in CSV: 86)
[9/131] 2021-12-22 to 2022-01-05: Saved 10 new articles (Total in CSV: 96)
[10/131] 2022-01-05 to 2022-01-19: Saved 12 new articles (Total in CSV: 108)
[11/131] 2022-01-19 to 2022-02-02: Saved 12 new articles (Total in CSV: 120)
[12/131] 2022-02-02 to 2022-02-16: Saved 13 new articles (Total in CSV: 133)
[13/131] 2022-02-16 to 2022-03-02: Saved 16 new articles (Total in CSV: 149)
[14/131] 2022-03-02 

## 6. Pembersihan Akhir dan Ringkasan Dataset

Langkah terakhir untuk merapikan dataset:
1. Menghapuskan baris yang kosong atau memiliki judul ganda (duplikat).
2. Mengurutkan berita secara kronologis berdasarkan tanggal rilisnya.
3. Menyimpan file CSV final dan menampilkan 10 sampel data teratas sebagai preview.

In [ ]:
df_final = pd.DataFrame(records)

if not df_final.empty:
    df_final = df_final.dropna(subset=["title", "content", "published_at"]).drop_duplicates(subset=["title"])
    df_final["published_at"] = pd.to_datetime(df_final["published_at"])
    df_final = df_final.sort_values(by="published_at").reset_index(drop=True)
    df_final.to_csv(SAVE_PATH, index=False)
    
    print("=" * 60)
    print(f"Dataset compiled! Saved {len(df_final)} verified articles to: {SAVE_PATH}[cite: 1]")
    print("=" * 60)
    
    preview = df_final[["published_at", "title", "source_domain"]].copy()
    preview["content_chars"] = df_final["content"].str.len()
    display(preview.head(10))
else:
    print("No records saved. Check network connection or query parameters.")

Dataset compiled! Saved 1118 verified articles to: ..\data/raw\geopolitical_news.csv[cite: 1]


,published_at,title,source_domain,content_chars
0,2015-09-25 09:48:25,Canadian Dollar,CNBC,5110
1,2021-09-01 07:00:00,U.S. relationship with Taliban unclear after e...,cnbc.com,3908
2,2021-09-01 07:00:00,Reserve Bank of India may begin policy normali...,cnbc.com,312
3,2021-09-04 07:00:00,'Stagflation' is the greatest threat to Europe...,cnbc.com,2177
4,2021-09-05 07:00:00,"Taliban, opposition fight for Afghan holdout p...",cnbc.com,5552
5,2021-09-05 07:00:00,The Indian rupee has had a stable run this yea...,cnbc.com,5299
6,2021-09-15 07:00:00,Hayman's Kyle Bass on the investor impact of C...,cnbc.com,2981
7,2021-09-15 07:00:00,"China sees 'Cold War mentality' in U.S., Briti...",cnbc.com,832
8,2021-09-16 07:00:00,Fraud stock? Scorpion issues scathing warning ...,cnbc.com,306
9,2021-09-16 07:00:00,Treasury sanctions Colombian drug trafficking ...,cnbc.com,1784
